# AASHTO LRFD — RC Columns and Traffic Railing Checks

Validation notebook for:
- **5.6.4.4** Axial resistance of compression members
- **5.6.4.2** Longitudinal reinforcement limits
- **5.6.4.6** Spiral reinforcement ratio
- **4.5.3.2.2b** Moment magnification (slenderness)
- **5.6.4.5** P-M interaction diagram (uniaxial, strain compatibility)
- **5.6.4.5** Biaxial flexure (moment-contour and Bresler reciprocal)
- **A13.3.1** Concrete parapet yield-line resistance
- **A13.4.2** Deck overhang collision tension
- **Table A13.2-1** Test-level design forces (TL-1 through TL-6)

Units: **kip, inch, ksi** throughout (matching the LRFD dimensional form).

Each section runs a worked example followed by a summary table of the `CheckResult` output.

In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd

from civilpy.structural.aashto.lrfd.columns import (
    RebarLayer,
    rc_column_axial_resistance,
    rc_column_reinforcement_limits,
    rc_spiral_reinforcement,
    moment_magnification,
    rc_pm_interaction_diagram,
    rc_pm_capacity_check,
    rc_biaxial_check,
)
from civilpy.structural.aashto.lrfd.railing import (
    TEST_LEVEL_LOADS,
    parapet_yield_line_capacity,
    parapet_test_level_check,
    deck_overhang_collision_tension,
)

## 1  Axial Resistance — 5.6.4.4

**Example:** 24 in × 24 in tied column, f'c = 4 ksi, fy = 60 ksi, 8-#9 bars (8 × 1.00 in²).

By hand:
- Ag = 576 in²
- Ast = 8.00 in²
- Po = 0.85(4)(576 − 8) + 60(8) = 1931.6 + 480 = 2411.6 kip
- Pn = 0.80 × 2411.6 = **1929.3 kip**
- φPn = 0.75 × 1929.3 = **1446.9 kip**

In [ ]:
col_ag = 24.0 * 24.0          # in²
col_ast = 8 * 1.00             # 8 #9 bars
fc = 4.0                       # ksi
fy = 60.0                      # ksi
pu_axial = 1200.0              # kip applied demand

axial = rc_column_axial_resistance(
    a_g=col_ag, a_st=col_ast, f_c=fc, f_y=fy, p_u=pu_axial, spiral=False
)
print(f"Po         = {axial.details['Po']:.1f} kip")
print(f"Pn (0.80Po) = {axial.capacity:.1f} kip")
print(f"φPn         = {axial.factored_capacity:.1f} kip")
print(f"DCR (Pu/φPn)= {pu_axial / axial.factored_capacity:.3f}")
print(f"OK?         = {axial.ok}")

## 2  Reinforcement Limits — 5.6.4.2

For the same 24×24 column with 8-#9 bars:
- ρ = 8.00/576 = 0.0139 ✓ (≤ 0.08)
- Ast·fy/(Ag·f'c) = 8×60/(576×4) = 480/2304 = 0.208 ✓ (≥ 0.135)

In [ ]:
limits = rc_column_reinforcement_limits(a_g=col_ag, a_st=col_ast, f_c=fc, f_y=fy)
print(f"ρ (Ast/Ag)           = {limits.details['Ast/Ag']:.4f}  (max 0.08)  {'✓' if limits.details['max_steel'] else '✗'}")
print(f"Ast·fy/(Ag·f'c)      = {limits.details['Ast*fy/(Ag*f\'c)']:.4f}  (min 0.135) {'✓' if limits.details['min_steel'] else '✗'}")
print(f"Overall OK?          = {limits.ok}")

## 3  Spiral Reinforcement — 5.6.4.6

**Example:** 24 in diameter circular spiral column.  
Core diameter (outside spiral) = 22 in → Dc = 22 in.  
#4 spiral (As = 0.20 in²) at 3 in pitch.  
f'c = 4 ksi, fy = 60 ksi.

By hand: ρs_min = 0.45(Ag/Ac − 1)(f'c/fy) = 0.45(452.4/380.1 − 1)(4/60) = 0.0114

Provided: ρs = 4·As/(Dc·s) = 4(0.20)/(22×3) = 0.0121 ✓

In [ ]:
from civilpy.structural.aashto.lrfd.columns import rc_spiral_reinforcement

spiral = rc_spiral_reinforcement(
    d_c=22.0,      # core diameter (in) — outside of spiral
    a_g=math.pi * 12.0**2,  # gross area of 24 in dia column
    a_sp=0.20,     # spiral bar area (in²)
    s=3.0,         # pitch (in)
    f_c=fc, f_y=fy,
)
print(f"ρs required = {spiral.details['rho_s_min']:.4f}")
print(f"ρs provided = {spiral.capacity:.4f}")
print(f"OK?         = {spiral.ok}")

## 4  Moment Magnification — 4.5.3.2.2b

**Example:** 24×24 tied column, Lu = 20 ft, M1 = 120 kip-ft, M2 = 280 kip-ft (single-curvature),
Pu = 600 kip, EI = 0.2·Ec·Ig (non-sway).

In [ ]:
ec = 57.0 * math.sqrt(fc * 1000.0)   # ksi
ig = 24.0**4 / 12.0                   # in⁴
ei = 0.2 * ec * ig                    # kip·in²

mag = moment_magnification(
    m1=120.0 * 12.0,     # kip-in
    m2=280.0 * 12.0,     # kip-in
    p_u=600.0,
    l_u=20.0 * 12.0,     # in
    ei=ei,
    k=1.0,               # pinned both ends (conservative)
    single_curvature=True,
)
print(f"Cm         = {mag.details['Cm']:.3f}")
print(f"Pc (Euler) = {mag.details['Pc']:.1f} kip")
print(f"δns        = {mag.details['delta_ns']:.3f}")
print(f"Mc = δns·M2= {mag.capacity:.1f} kip-in = {mag.capacity/12:.1f} kip-ft")

## 5  P-M Interaction Diagram — 5.6.4.5

**Example:** 24×24 tied column, f'c = 4 ksi, fy = 60 ksi.  
Longitudinal steel: 4 corner bars of 1.00 in² each, symmetrically placed at 2.5 in cover to bar center.

Layers (from top compression fiber):
- Top layer (2 bars): depth = 2.5 in, area = 2.00 in²
- Bottom layer (2 bars): depth = 21.5 in, area = 2.00 in²

In [ ]:
layers_rect = [
    RebarLayer(area=2.00, depth=2.5),   # top 2 bars
    RebarLayer(area=2.00, depth=21.5),  # bottom 2 bars
]

diagram = rc_pm_interaction_diagram(
    layers=layers_rect, f_c=fc, f_y=fy, h=24.0, b=24.0
)

phi_pn = [pt.phi_pn for pt in diagram]
phi_mn = [pt.phi_mn for pt in diagram]

fig, ax = plt.subplots(figsize=(6, 8))
ax.plot([m / 12.0 for m in phi_mn], phi_pn, 'b-', lw=2, label='φ·Interaction curve')

# demand points
demands = [(600, 1800), (800, 1200), (300, 2400)]   # (Pu kip, Mu kip-in)
for pu_d, mu_d in demands:
    ax.plot(mu_d / 12.0, pu_d, 'ro', ms=8)
    ax.annotate(f'({pu_d:.0f}k, {mu_d/12:.0f}k-ft)',
                (mu_d/12.0, pu_d), textcoords='offset points', xytext=(8, 4), fontsize=8)

ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('φMn (kip-ft)')
ax.set_ylabel('φPn (kip)')
ax.set_title('P-M Interaction — 24×24 Tied Column\nf\'c=4 ksi, fy=60 ksi, 4-#9')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Max φPn = {max(phi_pn):.1f} kip")
print(f"Max φMn = {max(phi_mn)/12:.1f} kip-ft")

### 5.1  P-M Capacity Check — demand point

In [ ]:
results = []
for pu_d, mu_d in demands:
    chk = rc_pm_capacity_check(
        p_u=pu_d, m_u=mu_d,
        layers=layers_rect, f_c=fc, f_y=fy, h=24.0, b=24.0
    )
    results.append({
        'Pu (kip)': pu_d,
        'Mu (kip-ft)': mu_d / 12,
        'φMn @ Pu (kip-ft)': round(chk.capacity / 12, 1),
        'Ratio': round(chk.ratio, 3) if chk.ratio else '—',
        'OK': '✓' if chk.ok else '✗'
    })

pd.DataFrame(results)

## 6  Circular Column P-M Diagram — 5.6.4.5

**Example:** 30 in diameter spiral column, f'c = 4 ksi, fy = 60 ksi.  
12-#8 bars (12 × 0.79 in²) uniformly distributed on a 25 in bolt circle.

Approximate layers from top of section at 2.5 in cover (centroid):
angles at 0°, 30°, 60°, 90°, 120°, 150°, 180°, 210°, 240°, 270°, 300°, 330°

In [ ]:
r_circ = 30.0 / 2.0     # section radius
r_bars = 25.0 / 2.0     # bolt circle radius
n_bars = 12
bar_area = 0.79          # #8

layers_circ = [
    RebarLayer(
        area=bar_area,
        depth=r_circ - r_bars * math.cos(math.radians(i * 360.0 / n_bars))
    )
    for i in range(n_bars)
]

diag_circ = rc_pm_interaction_diagram(
    layers=layers_circ, f_c=fc, f_y=fy, diameter=30.0, spiral=True
)

phi_pn_c = [pt.phi_pn for pt in diag_circ]
phi_mn_c = [pt.phi_mn for pt in diag_circ]

fig, ax = plt.subplots(figsize=(6, 8))
ax.plot([m / 12.0 for m in phi_mn_c], phi_pn_c, 'g-', lw=2, label='φ·Interaction (circular, spiral)')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('φMn (kip-ft)')
ax.set_ylabel('φPn (kip)')
ax.set_title('P-M Interaction — 30 in Circular Spiral Column\nf\'c=4 ksi, fy=60 ksi, 12-#8')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Max φPn = {max(phi_pn_c):.1f} kip")
print(f"Max φMn = {max(phi_mn_c)/12:.1f} kip-ft")

## 7  Biaxial Flexure — 5.6.4.5

Using the moment-contour method (low axial load) and the Bresler reciprocal method.

**Example (moment-contour):** Pu = 200 kip, Mux = 800 kip-in, Muy = 500 kip-in.  
From P-M checks: Mrx = 1200 kip-in, Mry = 900 kip-in at Pu.  
Utilization = 800/1200 + 500/900 = 0.667 + 0.556 = **1.22 ✗**

In [ ]:
# moment-contour (low axial)
biax_mc = rc_biaxial_check(
    p_u=200.0, m_ux=800.0, m_uy=500.0,
    m_rx=1200.0, m_ry=900.0,
    f_c=fc, a_g=col_ag,
)
print("Moment-contour method:")
print(f"  Mux/Mrx + Muy/Mry = {biax_mc.demand:.3f}  ({'✓' if biax_mc.ok else '✗'})")
print(f"  method: {biax_mc.details['method']}")
print()

# Bresler reciprocal (high axial)
biax_br = rc_biaxial_check(
    p_u=1000.0, m_ux=600.0, m_uy=400.0,
    m_rx=750.0, m_ry=600.0,   # uniaxial capacities at Pu
    p_rx=1350.0, p_ry=1200.0, # uniaxial axial capacities at eccentricities
    phi_p_o=1500.0,
)
print("Bresler reciprocal method:")
print(f"  Prxy = {biax_br.capacity:.1f} kip  vs  Pu = {biax_br.demand:.1f} kip  ({'✓' if biax_br.ok else '✗'})")
print(f"  method: {biax_br.details['method']}")

## 8  Table A13.2-1 — Test-Level Design Forces (NCHRP 350 Era)

Print the full table including the newly-added Minimum H (rail height) column.

In [ ]:
rows = []
for tl, d in TEST_LEVEL_LOADS.items():
    rows.append({
        'TL':        tl,
        'Ft (kip)':  d.f_t,
        'FL (kip)':  d.f_l,
        'Fv (kip)':  d.f_v,
        'Lt=LL (ft)': d.l_t,
        'Lv (ft)':   d.l_v,
        'He min (in)': d.h_e_min,
        'H min (in)':  d.h_min,
    })

df = pd.DataFrame(rows).set_index('TL')
df

## 9  Parapet Yield-Line Resistance — A13.3.1

**Example:** New Jersey-style concrete barrier, 32 in height (TL-4 target).
- Mc = 9.0 kip-ft/ft (wall flexural resistance per unit length, horizontal axis)
- Mw = 60.0 kip-ft (wall resistance about vertical axis)
- Mb = 0.0 (no top rail contribution)
- Lt = 3.5 ft (TL-4 distribution length)

By hand (interior region):
- Lc = 3.5/2 + √((3.5/2)² + 8·(32/12)·(0+60)/9) = 1.75 + √(3.0625 + 142.2) = 1.75 + 12.07 = **13.82 ft**
- Rw = (2/(2×13.82 − 3.5))·(8×0 + 8×60 + 9×13.82²/(32/12)) = (2/24.14)·(480 + 628.5) = **91.8 kip**

In [ ]:
m_c = 9.0    # kip-ft/ft
m_w = 60.0   # kip-ft
h_ft = 32.0 / 12.0  # 32 in → ft

# Interior check against TL-4
tl4 = parapet_test_level_check(
    test_level='TL-4',
    m_c=m_c, m_w=m_w, h_ft=h_ft,
    m_b=0.0, end_region=False,
)
print("TL-4 Interior region:")
print(f"  Lc  = {tl4.details['Lc']:.2f} ft")
print(f"  Rw  = {tl4.capacity:.1f} kip  (demand Ft = {tl4.demand:.0f} kip)")
print(f"  DCR = {tl4.demand/tl4.capacity:.3f}  {'✓' if tl4.ok else '✗'}")
print(f"  He check (≥ {TEST_LEVEL_LOADS['TL-4'].h_e_min}\"): {tl4.details['h_e_ok']}")
print(f"  H  check (≥ {TEST_LEVEL_LOADS['TL-4'].h_min}\"): {tl4.details['h_min_ok']}")

In [ ]:
# End region check
tl4_end = parapet_test_level_check(
    test_level='TL-4',
    m_c=m_c, m_w=m_w, h_ft=h_ft,
    m_b=0.0, end_region=True,
)
print("TL-4 End region:")
print(f"  Lc  = {tl4_end.details['Lc']:.2f} ft")
print(f"  Rw  = {tl4_end.capacity:.1f} kip  (demand Ft = {tl4_end.demand:.0f} kip)")
print(f"  DCR = {tl4_end.demand/tl4_end.capacity:.3f}  {'✓' if tl4_end.ok else '✗'}")

### 9.1  Railing capacity vs. all test levels

In [ ]:
rows_tl = []
for tl in TEST_LEVEL_LOADS:
    tld = TEST_LEVEL_LOADS[tl]
    for end in (False, True):
        chk = parapet_test_level_check(
            test_level=tl,
            m_c=m_c, m_w=m_w, h_ft=h_ft,
            m_b=0.0, end_region=end,
        )
        rows_tl.append({
            'TL':     tl,
            'Region': 'End' if end else 'Interior',
            'Lc (ft)': round(chk.details['Lc'], 2),
            'Rw (kip)': round(chk.capacity, 1),
            'Ft (kip)': tld.f_t,
            'DCR':    round(tld.f_t / chk.capacity, 3),
            'He ok':  chk.details['h_e_ok'],
            'H ok':   chk.details['h_min_ok'],
            'Pass':   '✓' if chk.ok else '✗',
        })

pd.DataFrame(rows_tl)

## 10  Deck Overhang Collision Tension — A13.4.2

For the TL-4 interior case above:
- Rw = Lc from TL-4 interior result, H = h_ft

In [ ]:
overhang = deck_overhang_collision_tension(
    r_w=tl4.capacity,
    l_c=tl4.details['Lc'],
    h=h_ft,
)
print(f"Distribution length = {overhang.details['distribution_length']:.2f} ft")
print(f"T (tension per ft)  = {overhang.capacity:.3f} kip/ft")
print(f"  (Rw / (Lc + 2H) = {tl4.capacity:.1f} / {overhang.details['distribution_length']:.2f})")

## 11  Summary

| Check | Article | Module | Status |
| --- | --- | --- | --- |
| Axial resistance | 5.6.4.4 | columns | Validated |
| Reinforcement limits | 5.6.4.2 | columns | Validated |
| Spiral reinforcement | 5.6.4.6 | columns | Validated |
| Moment magnification | 4.5.3.2.2b | columns | Validated |
| P-M interaction (rect) | 5.6.4.5 | columns | Validated |
| P-M interaction (circ) | 5.6.4.5 | columns | Validated |
| Biaxial flexure | 5.6.4.5 | columns | Validated |
| Test-level table | A13.2-1 | railing | Validated |
| Parapet yield-line | A13.3.1 | railing | Validated |
| Deck overhang tension | A13.4.2 | railing | Validated |